# Disaster Tweet Classification: Day 2 Preprocessing

## Goal
Build a clean and reproducible text preprocessing pipeline for tweet text.

## What this notebook does
1. Load `train.csv` and `test.csv`
2. Define `clean_text(text: str) -> str`
3. Show before/after preprocessing examples
4. Run a quick hashtag inspection
5. Apply preprocessing to full train and test data
6. Sanity-check the cleaned output

## Important note
This notebook does **not** train any model.
Modeling begins in `03_modeling_evaluation.ipynb`.

In [1]:
print("Imports and Constants")

import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"

SAMPLE_SIZE = 10
MIN_TOKEN_LENGTH = 2

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)

Imports and Constants


## NLTK resources

This project uses NLTK for three things:
- tokenization
- stopword removal
- lemmatization

These resources may need to be downloaded once on a new local machine before the notebook can run successfully.

In [2]:
print("NLTK (Natural Language Toolkit) Resources")

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

NLTK (Natural Language Toolkit) Resources


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kaust\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kaust\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kaust\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kaust\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kaust\AppData\Roaming\nltk_data...


True

In [3]:
print("Load Train and Test Data")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

Load Train and Test Data
Train shape: (7613, 5)
Test shape: (3263, 4)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation orders in California",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,1


,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, stay safe everyone."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are fleeing across the street, I cannot save them all"
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


## Why this preprocessing pipeline?

The project documentation defines the MVP preprocessing steps clearly:
- lowercase text
- remove URLs
- remove @mentions
- remove HTML entities
- remove non-alphabetic characters
- strip whitespace
- tokenize
- remove stopwords
- lemmatize
- rejoin cleaned tokens

For the MVP model, `text` is the main feature. `location` is too noisy, and `keyword` will be tested separately later if needed.

## Preprocessing components

The cleaning pipeline combines:
- regex rules for URLs, mentions, HTML artifacts, and non-letter characters
- NLTK stopwords for removing common low-information words
- WordNet lemmatization for reducing words to their base forms

This keeps the pipeline lightweight and appropriate for a classical NLP baseline.

In [4]:
print("Setup Preprocessing Tools")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

HTML_ENTITY_PATTERN = re.compile(r"&amp;|&lt;|&gt;")
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
MENTION_PATTERN = re.compile(r"@\w+")
NON_ALPHA_PATTERN = re.compile(r"[^a-zA-Z\s]")
EXTRA_SPACE_PATTERN = re.compile(r"\s+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")

Setup Preprocessing Tools


## Main cleaning function

The `clean_text()` function applies the preprocessing steps in a fixed order:
1. lowercase text
2. remove URLs
3. remove mentions
4. remove HTML entities
5. keep only letters and spaces
6. normalize whitespace
7. tokenize
8. remove stopwords
9. lemmatize
10. rejoin tokens

This function will become the main text normalization step used before TF-IDF vectorization in the next notebook.

In [5]:
print("Main Cleaning Function")

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    text = text.lower()
    text = re.sub(URL_PATTERN, " ", text)
    text = re.sub(MENTION_PATTERN, " ", text)
    text = re.sub(HTML_ENTITY_PATTERN, " ", text)
    text = re.sub(NON_ALPHA_PATTERN, " ", text)
    text = re.sub(EXTRA_SPACE_PATTERN, " ", text).strip()
    
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    tokens = [token for token in tokens if len(token) >= MIN_TOKEN_LENGTH]
    
    return " ".join(tokens)

Main Cleaning Function


## Step-by-step cleaning inspection

Before applying the pipeline to the full dataset, I inspect one tweet step by step.

This is useful because preprocessing bugs are often silent. A debug view helps verify that each rule is doing what I expect and not accidentally removing too much signal.

In [6]:
print("Debug function: Inspect each cleaning step")

def inspect_cleaning_steps(text: str):
    step_0 = str(text)
    step_1 = step_0.lower()
    step_2 = re.sub(URL_PATTERN, " ", step_1)
    step_3 = re.sub(MENTION_PATTERN, " ", step_2)
    step_4 = re.sub(HTML_ENTITY_PATTERN, " ", step_3)
    step_5 = re.sub(NON_ALPHA_PATTERN, " ", step_4)
    step_6 = re.sub(EXTRA_SPACE_PATTERN, " ", step_5).strip()
    step_7 = word_tokenize(step_6)
    step_8 = [token for token in step_7 if token not in stop_words]
    step_9 = [lemmatizer.lemmatize(token) for token in step_8]
    step_10 = [token for token in step_9 if len(token) >= MIN_TOKEN_LENGTH]
    final_text = " ".join(step_10)
    
    return {
        "original": step_0,
        "lowercase": step_1,
        "no_url": step_2,
        "no_mentions": step_3,
        "no_html_entities": step_4,
        "letters_only": step_5,
        "normalized_spaces": step_6,
        "tokens": step_7,
        "no_stopwords": step_8,
        "lemmatized": step_9,
        "final_tokens": step_10,
        "cleaned_text": final_text
    }

Debug function: Inspect each cleaning step


## Before vs after examples

A preprocessing pipeline should not be judged only by code correctness. It should also be checked qualitatively.

These examples help answer an important question: does the cleaned text still preserve the core meaning of the tweet after noise removal?

In [7]:
example_tweet = train_df.loc[0, "text"]
steps = inspect_cleaning_steps(example_tweet)

for key, value in steps.items():
    print(f"\n--- {key.upper()} ---")
    print(value)


--- ORIGINAL ---
Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all

--- LOWERCASE ---
our deeds are the reason of this #earthquake may allah forgive us all

--- NO_URL ---
our deeds are the reason of this #earthquake may allah forgive us all

--- NO_MENTIONS ---
our deeds are the reason of this #earthquake may allah forgive us all

--- NO_HTML_ENTITIES ---
our deeds are the reason of this #earthquake may allah forgive us all

--- LETTERS_ONLY ---
our deeds are the reason of this  earthquake may allah forgive us all

--- NORMALIZED_SPACES ---
our deeds are the reason of this earthquake may allah forgive us all

--- TOKENS ---
['our', 'deeds', 'are', 'the', 'reason', 'of', 'this', 'earthquake', 'may', 'allah', 'forgive', 'us', 'all']

--- NO_STOPWORDS ---
['deeds', 'reason', 'earthquake', 'may', 'allah', 'forgive', 'us']

--- LEMMATIZED ---
['deed', 'reason', 'earthquake', 'may', 'allah', 'forgive', 'u']

--- FINAL_TOKENS ---
['deed', 'reason', 'earthquake', 'may', '

## Hashtag checkpoint

The task list asks for a quick decision checkpoint on hashtags. This matters because hashtags often carry topical signal in tweet data.

In the current MVP pipeline, the `#` symbol is removed, but the underlying word is preserved. For example, `#wildfire` becomes `wildfire`. This keeps the lexical signal while simplifying the text.

In [8]:
print("Before/After Comparison of 10 sample tweets")

sample_df = train_df[["id", "text", "target"]].sample(SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
sample_df["cleaned_text"] = sample_df["text"].apply(clean_text)

for idx, row in sample_df.iterrows():
    print("=" * 120)
    print(f"ID: {row['id']} | TARGET: {row['target']}")
    print("\nBEFORE:")
    print(row["text"])
    print("\nAFTER:")
    print(row["cleaned_text"])
    print()

Before/After Comparison of 10 sample tweets
ID: 3796 | TARGET: 1

BEFORE:
So you have a new weapon that can cause un-imaginable destruction.

AFTER:
new weapon cause un imaginable destruction

ID: 3185 | TARGET: 0

BEFORE:
The f$&amp;@ing things I do for #GISHWHES Just got soaked in a deluge going for pads and tampons. Thx @mishacollins @/@

AFTER:
thing gishwhes got soaked deluge going pad tampon thx

ID: 7769 | TARGET: 1

BEFORE:
DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe CoL police can catch a pickpocket in Liverpool Stree... http://t.co/vXIn1gOq4Q

AFTER:
dt rt col police catch pickpocket liverpool stree

ID: 191 | TARGET: 0

BEFORE:
Aftershock back to school kick off was great. I want to thank everyone for making it possible. What a great night.

AFTER:
aftershock back school kick great want thank everyone making possible great night

ID: 9810 | TARGET: 0

BEFORE:
in response to trauma Children of Addicts develop a defensive self - one that decreases vulnerability. (3

AFTER:


In [9]:
display(sample_df)

,id,text,target,cleaned_text
2644,3796,So you have a new weapon that can cause un-imaginable destruction.,1,new weapon cause un imaginable destruction
2227,3185,The f$&amp;@ing things I do for #GISHWHES Just got soaked in a deluge going for pads and tampons. Thx @mishacollins @/@,0,thing gishwhes got soaked deluge going pad tampon thx
5448,7769,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe CoL police can catch a pickpocket in Liverpool Stree... http://t.co/vXIn1gOq4Q,1,dt rt col police catch pickpocket liverpool stree
132,191,Aftershock back to school kick off was great. I want to thank everyone for making it possible. What a great night.,0,aftershock back school kick great want thank everyone making possible great night
6845,9810,in response to trauma Children of Addicts develop a defensive self - one that decreases vulnerability. (3,0,response trauma child addict develop defensive self one decrease vulnerability
5559,7934,@Calum5SOS you look like you got caught in a rainstorm this is amazing and disgusting at the same time,0,look like got caught rainstorm amazing disgusting time
1765,2538,my favorite lady came to our volunteer meeting\nhopefully joining her youth collision and i am excite http://t.co/Ij0wQ490cS,1,favorite lady came volunteer meeting hopefully joining youth collision excite
1817,2611,@brianroemmele UX fail of EMV - people want to insert and remove quickly like a gas pump stripe reader. 1 person told me it crashed the POS,1,ux fail emv people want insert remove quickly like gas pump stripe reader person told crashed po
6810,9756,Can't find my ariana grande shirt this is a fucking tragedy,0,find ariana grande shirt fucking tragedy
4398,6254,The Murderous Story Of AmericaÛªs First Hijacking http://t.co/EYUGk6byxr,1,murderous story america first hijacking


## Hashtag checkpoint

The project task list explicitly asks for a quick decision checkpoint on whether hashtags help or hurt.  
This check is not a full experiment yet — it is just a quick inspection of hashtag frequency in raw tweets.

In [10]:
print("Hashtag Extraction Helper")

def extract_hashtags(text: str):
    if pd.isna(text):
        return []
    return [tag.lower() for tag in HASHTAG_PATTERN.findall(str(text))]

Hashtag Extraction Helper


In [11]:
print("Extraction helper from Raw Train Text")

train_df["hashtags"] = train_df["text"].apply(extract_hashtags)
train_df["has_hashtag"] = train_df["hashtags"].apply(lambda x: len(x) > 0)

hashtag_summary = pd.DataFrame({
    "tweets_with_hashtag": train_df.groupby("target")["has_hashtag"].sum(),
    "total_tweets": train_df.groupby("target")["has_hashtag"].count()
})

hashtag_summary["percent_with_hashtag"] = (
    hashtag_summary["tweets_with_hashtag"] / hashtag_summary["total_tweets"] * 100
).round(2)

display(hashtag_summary)

Extraction helper from Raw Train Text


,tweets_with_hashtag,total_tweets,percent_with_hashtag
target,,,
0,885,4342,20.38
1,858,3271,26.23


In [12]:
print("Top Hashtags by Class")

non_disaster_hashtags = Counter(
    tag
    for tags in train_df.loc[train_df["target"] == 0, "hashtags"]
    for tag in tags
)

disaster_hashtags = Counter(
    tag
    for tags in train_df.loc[train_df["target"] == 1, "hashtags"]
    for tag in tags
)

top_non_disaster_hashtags = pd.DataFrame(
    non_disaster_hashtags.most_common(15),
    columns=["hashtag", "count"]
)

top_disaster_hashtags = pd.DataFrame(
    disaster_hashtags.most_common(15),
    columns=["hashtag", "count"]
)

print("Top hashtags — Not Disaster")
display(top_non_disaster_hashtags)

print("Top hashtags — Disaster")
display(top_disaster_hashtags)

Top Hashtags by Class
Top hashtags — Not Disaster


,hashtag,count
0,nowplaying,21
1,news,20
2,hot,18
3,prebreak,17
4,best,17
5,gbbo,14
6,jobs,14
7,islam,14
8,job,12
9,hiring,10


Top hashtags — Disaster


,hashtag,count
0,news,56
1,hiroshima,22
2,earthquake,19
3,hot,13
4,prebreak,13
5,best,13
6,japan,11
7,india,10
8,yyc,10
9,breaking,9


In [13]:
print("Hashtag Decision Note")

print("Decision checkpoint:")
print("- Hashtags clearly contain topic signal in raw text.")
print("- For the current clean_text() MVP pipeline, the '#' symbol is removed but the word remains.")
print("- Example: #wildfire becomes wildfire.")
print("- That means we keep the core lexical signal without preserving hashtag formatting.")
print("- A future experiment can compare raw-hashtag preservation vs current cleaning.")

Hashtag Decision Note
Decision checkpoint:
- Hashtags clearly contain topic signal in raw text.
- For the current clean_text() MVP pipeline, the '#' symbol is removed but the word remains.
- Example: #wildfire becomes wildfire.
- That means we keep the core lexical signal without preserving hashtag formatting.
- A future experiment can compare raw-hashtag preservation vs current cleaning.


## Apply preprocessing to full datasets

Once the cleaning function looks correct on sample tweets, I apply it to the full training and test datasets.

I store the output in a new `cleaned_text` column rather than overwriting the original `text` column, so the raw text remains available for debugging and later comparison.

In [14]:
print("Applying preprocessing to full train and test datasets")

train_df["cleaned_text"] = train_df["text"].apply(clean_text)
test_df["cleaned_text"] = test_df["text"].apply(clean_text)

print("Preprocessing applied to full train and test datasets.")

Applying preprocessing to full train and test datasets
Preprocessing applied to full train and test datasets.


## Sanity check: how much text changed?

After preprocessing, I compare raw and cleaned text lengths.

This gives a quick quantitative sense of how aggressive the cleaning pipeline is. If cleaned text becomes too short too often, that may indicate signal loss. If it barely changes, the pipeline may be too weak.

In [15]:
print("Sanity Check on Cleaned Text")

print("Missing cleaned_text in train:", train_df["cleaned_text"].isna().sum())
print("Missing cleaned_text in test:", test_df["cleaned_text"].isna().sum())

print("\nEmpty cleaned_text rows in train:", (train_df["cleaned_text"].str.len() == 0).sum())
print("Empty cleaned_text rows in test:", (test_df["cleaned_text"].str.len() == 0).sum())

Sanity Check on Cleaned Text
Missing cleaned_text in train: 0
Missing cleaned_text in test: 0

Empty cleaned_text rows in train: 2
Empty cleaned_text rows in test: 1


In [16]:
print("Comparing Raw vs Cleaned Length")

train_df["raw_char_count"] = train_df["text"].astype(str).apply(len)
train_df["cleaned_char_count"] = train_df["cleaned_text"].astype(str).apply(len)
train_df["raw_word_count"] = train_df["text"].astype(str).apply(lambda x: len(x.split()))
train_df["cleaned_word_count"] = train_df["cleaned_text"].astype(str).apply(lambda x: len(x.split()))

length_compare = train_df[[
    "raw_char_count",
    "cleaned_char_count",
    "raw_word_count",
    "cleaned_word_count"
]].describe().T.round(2)

display(length_compare)

Comparing Raw vs Cleaned Length


,count,mean,std,min,25%,50%,75%,max
raw_char_count,7613.0,101.04,33.78,7.0,78.0,107.0,133.0,157.0
cleaned_char_count,7613.0,57.66,23.82,0.0,40.0,58.0,76.0,137.0
raw_word_count,7613.0,14.90,5.73,1.0,11.0,15.0,19.0,31.0
cleaned_word_count,7613.0,8.58,3.47,0.0,6.0,9.0,11.0,23.0


In [17]:
cleaned_examples = train_df[["id", "text", "cleaned_text", "target"]].sample(10, random_state=RANDOM_STATE + 7)
display(cleaned_examples)

,id,text,cleaned_text,target
74,107,I-77 Mile Marker 31 South Mooresville Iredell Vehicle Accident Ramp Closed at 8/6 1:18 PM,mile marker south mooresville iredell vehicle accident ramp closed pm,1
5387,7687,tomorrow's going to be a year since I went to the Panic! concert dressed as afycso ryan do u guys remember that,tomorrow going year since went panic concert dressed afycso ryan guy remember,1
4259,6051,Arnhem Weather - &lt;p&gt;An unrelenting and dangerous heat wave will expand across the South Central United StatesÛ_ http://t.co/yhAqa5WXoK,arnhem weather unrelenting dangerous heat wave expand across south central united state,1
2119,3045,Y'all PUSSSSSSSSSY AND SHOOOK TO DEATH OF ME,pusssssssssy shoook death,0
6078,8684,Georgia sinkhole closes road swallows whole pond http://t.co/cPEQv52LNA,georgia sinkhole close road swallow whole pond,1
37,55,INEC Office in Abia Set Ablaze - http://t.co/3ImaomknnA,inec office abia set ablaze,1
2118,3044,Afghan peace talks in doubt after Mullah Omar's death - Financial Times | #Mullah,afghan peace talk doubt mullah omar death financial time mullah,0
3931,5589,Internet basics: the flood defective intertissue is soul mate the milk trench host: GUAbxFv http://t.co/uzsQzYcB8X,internet basic flood defective intertissue soul mate milk trench host guabxfv,1
530,770,I saw two great punk bands making original music last week. Check em out here!! @GHOSTOFTHEAV @MontroseBand https://t.co/WdvxjsQwic,saw two great punk band making original music last week check em,0
4186,5947,@phiddleface NOT IF THERES A CHOKING HAZARD!!! ???? dont die before i get there!!,there choking hazard dont die get,0


## Keyword decoding for future experiments

The `keyword` column is not part of the MVP baseline, but it is still worth lightly cleaning now.

Some values are URL-encoded (for example, `%20` instead of spaces), so decoding them makes later experiments easier and prevents inconsistent keyword forms from leaking into feature engineering.

In [18]:
print("Lightweight keyword decoding helper")

def decode_keyword(keyword):
    if pd.isna(keyword):
        return np.nan
    return str(keyword).replace("%20", " ").lower().strip()

train_df["keyword_decoded"] = train_df["keyword"].apply(decode_keyword)
test_df["keyword_decoded"] = test_df["keyword"].apply(decode_keyword)

display(train_df[["keyword", "keyword_decoded"]].dropna().drop_duplicates().head(20))

Lightweight keyword decoding helper


,keyword,keyword_decoded
31,ablaze,ablaze
67,accident,accident
102,aftershock,aftershock
136,airplane%20accident,airplane accident
171,ambulance,ambulance
209,annihilated,annihilated
243,annihilation,annihilation
272,apocalypse,apocalypse
304,armageddon,armageddon
346,army,army


In [19]:
print("Final Column Check")

print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Final Column Check
Train columns:
['id', 'keyword', 'location', 'text', 'target', 'hashtags', 'has_hashtag', 'cleaned_text', 'raw_char_count', 'cleaned_char_count', 'raw_word_count', 'cleaned_word_count', 'keyword_decoded']

Test columns:
['id', 'keyword', 'location', 'text', 'cleaned_text', 'keyword_decoded']


## Day 2 Observations

### 1. Cleaning effect
- The preprocessing pipeline successfully removed URLs, mentions, punctuation noise, and HTML artifacts.
- The cleaned text is much shorter and more standardized than the raw tweet text.
- Lemmatization reduced word-form variation, which should help classical sparse-text models generalize better.

### 2. Feature decision for MVP
- `cleaned_text` will be the primary input feature for the baseline model.
- I will not use `location` in the MVP because it is noisy and highly incomplete.
- I decoded `keyword` values for future experiments, but I will not include `keyword` in the first baseline because of leakage risk.

### 3. Hashtag takeaway
- Hashtags contain useful topic signal in tweet text.
- The current pipeline removes the `#` symbol but keeps the underlying word, which is a reasonable MVP compromise.
- Later, I can test whether explicitly preserving hashtag formatting improves performance.

### 4. Tradeoff noticed during cleaning
- Some tweets become very compact after preprocessing, especially when they contain links, mentions, or formatting noise.
- This is expected, but it also means I should watch out for over-cleaning when evaluating model errors later.

### 5. Next step
- The next notebook will use `cleaned_text` for the baseline model.
- I will split the training data first and fit TF-IDF only on `X_train` to avoid leakage.
- The first baseline will be TF-IDF unigrams with Multinomial Naive Bayes.